# RoadScan — Published RDD2022 weights

This separate experiment evaluates SreekarAditya’s published YOLOv8s checkpoint without additional training. It uses GPU for evaluation and CPU for a separate latency measurement. All weights, reports and plots are saved on Kaggle.

## 1. Verify the data and published checkpoint

Reuse the exact 1,977 China MotorBike images and labels saved by the successful team v2 training notebook; verify every image hash against its saved manifest. Download the published checkpoint, verify its SHA-256 and confirm the four class IDs: D00, D10, D20 and D40. Save its source, license, training arguments and evaluation protocol. The cell prints an interpretation after these checks.

**Evaluation limit:** this model was already trained on RDD2022. We do not have its image-level training manifest, so these images are not verified independent tests. This notebook provides diagnostic results only. US and local test photos are not used.

Sources: [published model and class definitions](https://huggingface.co/SreekarAditya/yolo-rdd2022-benchmark), [checkpoint and SHA-256](https://huggingface.co/SreekarAditya/yolo-rdd2022-benchmark/blob/main/yolo-rdd2022-benchmark/yolov8s_seed0_best.pt).

In [1]:
from pathlib import Path
from collections import Counter
import sys, subprocess, importlib.metadata, hashlib, json, math, shutil, time, platform
import urllib.request, zipfile, pickletools
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, Markdown
import yaml

ART = Path('/kaggle/working/artifacts_rdd_pretrained')
ART.mkdir(exist_ok=True)
CLASSES = ['D00','D10','D20','D40']
SOURCE = Path('/kaggle/input/notebooks/rimasabunasser/14-roadscan-team-v2-training/team_v2_data')
ROOT = Path('/kaggle/working/rdd_pretrained_eval')
for folder in ['images','labels']:
    (ROOT/'val'/folder).mkdir(parents=True,exist_ok=True)
images = sorted((SOURCE/'val/images').glob('*.jpg'))
assert len(images)==1977 and all(p.stem.startswith('China_MotorBike_') for p in images)
rows=[]
for i,p in enumerate(images,1):
    label=SOURCE/'val/labels'/(p.stem+'.txt')
    assert label.is_file(),str(label)
    with Image.open(p) as im:
        im.load()
    present=[0]*4
    for line in label.read_text().splitlines():
        if not line.strip(): continue
        values=list(map(float,line.split()))
        assert len(values)==5 and all(math.isfinite(x) for x in values)
        c,x,y,w,h=values
        assert c==int(c) and 0<=c<4 and 0<w<=1 and 0<h<=1
        assert x-w/2>=-1e-4 and y-h/2>=-1e-4 and x+w/2<=1.0001 and y+h/2<=1.0001
        present[int(c)]=1
    shutil.copy2(p,ROOT/'val/images'/p.name)
    shutil.copy2(label,ROOT/'val/labels'/label.name)
    rows.append(dict(stem=p.stem,country='China_MotorBike',sha256=hashlib.sha256(p.read_bytes()).hexdigest(),**dict(zip(CLASSES,present))))
    if i%500==0: print('Audited validation images:',i,flush=True)
manifest=pd.DataFrame(rows)
frozen=pd.read_csv(SOURCE.parent/'artifacts_team_v2/manifest.csv')
expected=frozen.loc[frozen['split'].eq('val')].set_index('stem')['sha256'].to_dict()
assert manifest.set_index('stem')['sha256'].to_dict()==expected, 'Validation images differ from frozen team v2 manifest.'
manifest.to_csv(ART/'diagnostic_manifest.csv',index=False)
DATA=ROOT/'data.yaml'
DATA.write_text(yaml.safe_dump(dict(path=str(ROOT),train='val/images',val='val/images',names=dict(enumerate(CLASSES))),sort_keys=False))
# train is a required YAML field only; this notebook never calls model.train().

MODEL_CARD='https://huggingface.co/SreekarAditya/yolo-rdd2022-benchmark'
WEIGHT_URL='https://huggingface.co/SreekarAditya/yolo-rdd2022-benchmark/resolve/main/yolo-rdd2022-benchmark/yolov8s_seed0_best.pt?download=true'
EXPECTED_SHA256='565aae6915e34e4f7f4c44dac0bdcf9fff21b83c4adc74cc97ca09cc31882ce8'
WEIGHTS=ART/'yolov8s_seed0_best.pt'
if not WEIGHTS.exists():
    request=urllib.request.Request(WEIGHT_URL,headers={'User-Agent':'RoadScan-research','Cache-Control':'no-cache'})
    temporary=WEIGHTS.with_suffix('.download')
    with urllib.request.urlopen(request,timeout=180) as response, temporary.open('wb') as output:
        shutil.copyfileobj(response,output)
    assert hashlib.sha256(temporary.read_bytes()).hexdigest()==EXPECTED_SHA256,'Checkpoint digest mismatch; do not load.'
    temporary.replace(WEIGHTS)
assert hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()==EXPECTED_SHA256
# Inspect pickle globals without executing or unpickling checkpoint content.
with zipfile.ZipFile(WEIGHTS) as archive:
    pickle_name=next(n for n in archive.namelist() if n.endswith('/data.pkl'))
    globals_used=sorted({arg for op,arg,pos in pickletools.genops(archive.read(pickle_name)) if op.name=='GLOBAL'})
(ART/'checkpoint_globals.json').write_text(json.dumps(globals_used,indent=2))
try:
    installed=importlib.metadata.version('ultralytics')
except importlib.metadata.PackageNotFoundError:
    installed=None
if installed!='8.4.177':
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.177'])
import torch, ultralytics
from ultralytics import YOLO
# GPU diagnostic evaluation; CPU latency is measured separately.
torch.set_num_threads(2)
assert torch.cuda.is_available(), 'Enable GPU T4 x2.'
external=YOLO(str(WEIGHTS))
actual_names=[str(external.names[i]) for i in range(len(external.names))]
expected_labels=['longitudinal_crack','transverse_crack','alligator_crack','pothole']
assert actual_names in [CLASSES,expected_labels],('Unexpected class order; stop and review:',actual_names)
class_mapping=[dict(id=i,published_name=actual_names[i],rdd_code=CLASSES[i]) for i in range(4)]
(ART/'verified_class_mapping.json').write_text(json.dumps(class_mapping,indent=2))
display(pd.DataFrame(class_mapping))
args=getattr(external.model,'args',{})
(ART/'checkpoint_training_args.json').write_text(json.dumps(args,indent=2,default=str))
protocol=dict(model='YOLOv8s RDD2022 seed0',source=MODEL_CARD,download=WEIGHT_URL,sha256=EXPECTED_SHA256,license='CC BY 4.0 as stated by publisher; attribute SreekarAditya and RDD2022',classes=actual_names,evaluation_images=1977,evaluation_country='China_MotorBike',evaluation_status='diagnostic only; external pretraining overlap is not excluded',external_training_manifest_available=False,clean_generalization_claim_allowed=False,comparison_to_team='descriptive only; different external training data and architecture',training_performed=False,threshold_tuning=False,local_test_used=False,US_test_used=False,device='cuda:0',imgsz=640,ultralytics=ultralytics.__version__,torch=torch.__version__,python=platform.python_version())
(ART/'provenance_and_protocol.json').write_text(json.dumps(protocol,indent=2))
display(pd.DataFrame([dict(images=len(manifest),**{c:int(manifest[c].sum()) for c in CLASSES})]))
display(Markdown('**Preparation interpretation.** All 1,977 China MotorBike images passed decoding and label checks. The downloaded checkpoint matches the published SHA-256 and its class order is exactly D00, D10, D20, D40. This is an externally trained RDD2022 model: overlap with these images has not been excluded, so the next results are diagnostic, not an independent test. No training, US evaluation, local evaluation or threshold tuning is performed.'))
print('EXTERNAL_MODEL_READY',flush=True)


Audited validation images: 500
Audited validation images: 1000
Audited validation images: 1500
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


,id,published_name,rdd_code
0,0,longitudinal_crack,D00
1,1,transverse_crack,D10
2,2,alligator_crack,D20
3,3,pothole,D40


,images,D00,D10,D20,D40
0,1977,1419,738,486,164


**Preparation interpretation.** All 1,977 China MotorBike images passed decoding and label checks. The downloaded checkpoint matches the published SHA-256 and its class order is exactly D00, D10, D20, D40. This is an externally trained RDD2022 model: overlap with these images has not been excluded, so the next results are diagnostic, not an independent test. No training, US evaluation, local evaluation or threshold tuning is performed.

EXTERNAL_MODEL_READY


## 2. Evaluate the published model without further training

Run the unchanged checkpoint on all 1,977 diagnostic images at 640 pixels using GPU. Report detection mAP@50, mAP@50–95, per-class AP, confusion plots and image-level precision, recall and Macro-F1 at a fixed 0.5 threshold. Measure detector CPU latency on 20 images after warm-up.

The code saves the weights, metrics, scores, plots and an English interpretation generated from the actual results. Detection AP and image-level F1 are different measures. No threshold search or fine-tuning is done in this experiment.

**Decision rule:** use this run to inspect model behaviour and missed damage. Before selecting or fine-tuning it, establish independent development data outside its pretraining images. Keep local photos for the final frozen-model test.

In [2]:
from sklearn.metrics import precision_recall_fscore_support
print('Starting full 1,977-image GPU diagnostic evaluation.',flush=True)
diagnostic=external.val(data=str(DATA),split='val',imgsz=640,batch=16,device=0,workers=2,conf=0.001,iou=0.7,max_det=300,plots=True,project=str(ART/'detection'),name='china_diagnostic',exist_ok=True)
detection_metrics={str(k):float(v) for k,v in diagnostic.results_dict.items()}
per_class=pd.DataFrame(dict(cls=CLASSES,AP50=[float(v) for v in diagnostic.box.ap50],AP50_95=[float(v) for v in diagnostic.box.ap]))
per_class.to_csv(ART/'diagnostic_per_class_ap.csv',index=False)
display(per_class)
part=manifest.sort_values('stem').reset_index(drop=True)
stems=part.stem.tolist()
lookup={s:i for i,s in enumerate(stems)}
truth=part[CLASSES].to_numpy(dtype=int)
scores=np.zeros((len(stems),4),dtype=float)
for n,result in enumerate(external.predict(source=str(ROOT/'val/images'),imgsz=640,conf=0.001,iou=0.7,max_det=300,device=0,stream=True,verbose=False),1):
    i=lookup[Path(result.path).stem]
    for c,p in zip(result.boxes.cls.cpu().numpy().astype(int),result.boxes.conf.cpu().numpy()):
        scores[i,c]=max(scores[i,c],float(p))
    if n%250==0: print('Image-level scores:',n,'/',len(stems),flush=True)
p,r,f,support=precision_recall_fscore_support(truth,scores>=0.5,zero_division=0)
image_metrics=pd.DataFrame(dict(cls=CLASSES,threshold=0.5,precision=p,recall=r,f1=f,support=support))
image_metrics.to_csv(ART/'diagnostic_image_metrics_fixed_0.5.csv',index=False)
np.savez_compressed(ART/'diagnostic_image_scores.npz',stems=np.array(stems),truth=truth,scores=scores)
display(image_metrics)
# Representative deterministic latency sample; excludes model loading and full app overhead.
cpu_model=YOLO(str(WEIGHTS))
rng=np.random.default_rng(42)
latency_images=[ROOT/'val/images'/(stems[int(i)]+'.jpg') for i in rng.choice(len(stems),20,replace=False)]
for _ in range(3):
    cpu_model.predict(str(latency_images[0]),imgsz=640,device='cpu',conf=0.25,verbose=False)
assert next(cpu_model.model.parameters()).device.type=='cpu', 'CPU latency measurement must use CPU.'
latencies=[]
for image in latency_images:
    start=time.perf_counter()
    cpu_model.predict(str(image),imgsz=640,device='cpu',conf=0.25,verbose=False)
    latencies.append(time.perf_counter()-start)
pd.DataFrame(dict(image=[p.name for p in latency_images],seconds=latencies)).to_csv(ART/'cpu_latency.csv',index=False)
summary=dict(protocol=protocol,detection=detection_metrics,image_macro_f1_fixed_0_5=float(f.mean()),cpu_latency=dict(images=20,warmup=3,threads=2,median_seconds=float(np.median(latencies)),p95_seconds=float(np.percentile(latencies,95))),fine_tuning_performed=False,independent_evaluation_pending=True)
(ART/'diagnostic_summary.json').write_text(json.dumps(summary,indent=2))
weak=per_class.sort_values('AP50').iloc[0]
interpretation=(f"### Diagnostic results — interpretation\n\n"
    f"Detection mAP@50 is **{detection_metrics['metrics/mAP50(B)']:.2%}** and mAP@50–95 is **{detection_metrics['metrics/mAP50-95(B)']:.2%}**. "
    f"The weakest detection class is **{weak['cls']}**, with AP50 **{weak['AP50']:.2%}**. "
    f"Image-level Macro-F1 at the fixed 0.5 threshold is **{f.mean():.2%}**; it measures image damage presence, not box accuracy. "
    f"Median CPU detector latency is **{np.median(latencies):.3f} seconds/image** on a 20-image sample, excluding the rest of the app.\n\n"
    "**These are diagnostic RDD2022 results. External training overlap has not been excluded. High scores do not prove generalization or a fair improvement over the team model.** "
    "No training or confidence tuning was performed. The next decision requires documented non-overlapping development data. "
    "Only then consider fine-tuning; keep the 40–80 local images for the frozen final test, never for selecting checkpoints or thresholds.")
display(Markdown(interpretation))
(ART/'RESULTS_INTERPRETATION.md').write_text(interpretation)
print('DIAGNOSTIC_COMPLETE',json.dumps(summary),flush=True)


Starting full 1,977-image GPU diagnostic evaluation.
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1872.7±414.8 MB/s, size: 76.9 KB)
val: Scanning /kaggle/working/rdd_pretrained_eval/val/labels... 1977 images, 43 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1977/1977 1.3Kit/s 1.5s
val: New cache created: /kaggle/working/rdd_pretrained_eval/val/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 124/124 4.7it/s 26.5s
                   all       1977       4650      0.924      0.924      0.962      0.671
    longitudinal_crack       1419       2678      0.904      0.901      0.948      0.678
      transverse_crack        738       1096      0.888      0.867      0.934      0.619
       alligator_crack        486        641      0.939      0.964      0.981

,cls,AP50,AP50_95
0,D00,0.947927,0.678260
1,D10,0.933620,0.618771
2,D20,0.980658,0.744731
3,D40,0.985648,0.643252


Image-level scores: 250 / 1977
Image-level scores: 500 / 1977
Image-level scores: 750 / 1977
Image-level scores: 1000 / 1977
Image-level scores: 1250 / 1977
Image-level scores: 1500 / 1977
Image-level scores: 1750 / 1977


,cls,threshold,precision,recall,f1,support
0,D00,0.5,0.998451,0.908386,0.951292,1419
1,D10,0.5,0.994941,0.799458,0.886551,738
2,D20,0.5,0.964948,0.962963,0.963955,486
3,D40,0.5,0.993711,0.963415,0.978328,164


### Diagnostic results — interpretation

Detection mAP@50 is **96.20%** and mAP@50–95 is **67.13%**. The weakest detection class is **D10**, with AP50 **93.36%**. Image-level Macro-F1 at the fixed 0.5 threshold is **94.50%**; it measures image damage presence, not box accuracy. Median CPU detector latency is **0.194 seconds/image** on a 20-image sample, excluding the rest of the app.

**These are diagnostic RDD2022 results. External training overlap has not been excluded. High scores do not prove generalization or a fair improvement over the team model.** No training or confidence tuning was performed. The next decision requires documented non-overlapping development data. Only then consider fine-tuning; keep the 40–80 local images for the frozen final test, never for selecting checkpoints or thresholds.

DIAGNOSTIC_COMPLETE {"protocol": {"model": "YOLOv8s RDD2022 seed0", "source": "https://huggingface.co/SreekarAditya/yolo-rdd2022-benchmark", "download": "https://huggingface.co/SreekarAditya/yolo-rdd2022-benchmark/resolve/main/yolo-rdd2022-benchmark/yolov8s_seed0_best.pt?download=true", "sha256": "565aae6915e34e4f7f4c44dac0bdcf9fff21b83c4adc74cc97ca09cc31882ce8", "license": "CC BY 4.0 as stated by publisher; attribute SreekarAditya and RDD2022", "classes": ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole"], "evaluation_images": 1977, "evaluation_country": "China_MotorBike", "evaluation_status": "diagnostic only; external pretraining overlap is not excluded", "external_training_manifest_available": false, "clean_generalization_claim_allowed": false, "comparison_to_team": "descriptive only; different external training data and architecture", "training_performed": false, "threshold_tuning": false, "local_test_used": false, "US_test_used": false, "device": "cuda:0", "